# CTMC MLE training (DeepSets model_0929)

`train.ipynb` の CTMCMLELoss を使った学習 / ファインチューニング用ノートブックです。
- まず生データを読み込んでデータローダを組み立てます。
- CTMCMLELoss をシーケンス全体に対して適用するヘルパを用意します。
- 学習/検証ループは関数化して、初期学習と既存モデルのファインチューニングの両方で使います。


In [ ]:
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

# プロジェクト内モジュール
from models.model_0929 import DeepSets_varSets_forDiagnel, collate_fn, varSets_Datasets
from utils.formate_matrix_toMLData import formate_dataMatrix, matrix_trimer

# デバイス設定
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

device

In [ ]:
# データ読み込み用パラメータ
parent_path = Path("/media/user/TRANSCEND/datas/")  # 適宜変更
# dir_list = ["data1", "data2", "data3", "data4", "data5"]
# 離散データを使う場合は以下のように切り替え
dir_list = ["discrete_data1", "discrete_data2", "discrete_data3", "discrete_data4", "discrete_data5"]

# ハイパーパラメータ
batch_size = 128
num_epochs = 300
lr = 1e-3
patience = 20  # 早期終了

# 保存先
scratch_ckpt = Path("model_weights/ctmc_mle/latest.pt")
finetune_ckpt = Path("model_weights/ctmc_mle/finetuned.pt")
pretrained_path = Path("model_weights/mixed_distribution/mixed_0926_2.pth")  # 既存モデル（必要に応じて変更）


In [ ]:
# データ読み込みヘルパ
VALID_EXTENSIONS = (".csv", ".txt")
IGNORED_PREFIXES = ("._", ".DS_Store", "Thumbs.db")
formater = formate_dataMatrix()

states, del_t, outputs = [], [], []

def process_file(file_path: Path):
    try:
        print(f"Processing: {file_path}")
        with open(file_path, "rb") as f:
            all_matrix = np.loadtxt(f, delimiter=",")

        tm = matrix_trimer(all_matrix)
        trm = tm.trim_transitionRateMatrix()
        data = tm.trim_data()
        output_vec = np.array(formater.GetOutputVector_byDiagonal(trm))

        # state: shape (2, num_samples_i)
        state = np.stack([data[:, 0], data[:, 1]], axis=0)
        states.append(state)
        del_t.append(data[:, 2])
        outputs.append(output_vec)
    except Exception as e:
        print(f"❌ Skipping file: {file_path} (Reason: {e})")


def process_all_files_in_directory(directory: Path):
    for filename in os.listdir(directory):
        if filename.startswith(IGNORED_PREFIXES):
            continue
        if not filename.endswith(VALID_EXTENSIONS):
            continue
        file_path = directory / filename
        if file_path.is_file():
            process_file(file_path)


for dir_name in dir_list:
    process_all_files_in_directory(parent_path / dir_name)

print(f"Total data size: {len(states)}")

In [ ]:
# Train/Val split と DataLoader 準備
train_states, val_states, train_del_t, val_del_t, train_outputs, val_outputs = train_test_split(
    states, del_t, outputs, test_size=0.1, random_state=42
)

train_dataset = varSets_Datasets(train_states, train_del_t, train_outputs)
val_dataset = varSets_Datasets(val_states, val_del_t, val_outputs)

use_cuda = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=use_cuda,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=use_cuda,
)

len(train_dataset), len(val_dataset)

In [ ]:
class CTMCMLELoss(nn.Module):
    '''
    Implements the true CTMC negative log-likelihood loss:
        L = -log [exp(Q t)]_{i, j}
    for each observation (i, j, t).
    '''
    def __init__(self, eps: float = 1e-12):
        super().__init__()
        self.eps = eps

    def forward(self,
                q_pred: torch.Tensor,   # shape: (batch, 3) → q12, q23, q34
                i_obs: torch.Tensor,    # shape: (batch,)
                j_obs: torch.Tensor,    # shape: (batch,)
                t_obs: torch.Tensor):   # shape: (batch,)

        batch = q_pred.shape[0]

        # clamp to avoid numerical issues
        q12 = q_pred[:, 0].clamp_min(self.eps)
        q23 = q_pred[:, 1].clamp_min(self.eps)
        q34 = q_pred[:, 2].clamp_min(self.eps)

        # Construct Q matrices for each sample
        Q = torch.zeros((batch, 4, 4), dtype=q_pred.dtype, device=q_pred.device)
        Q[:, 0, 1] = q12
        Q[:, 1, 2] = q23
        Q[:, 2, 3] = q34
        Q[:, 0, 0] = -q12
        Q[:, 1, 1] = -q23
        Q[:, 2, 2] = -q34
        Q[:, 3, 3] = 0.0   # absorbing state

        # Compute P(t) = exp(Q t) for each sample
        P_list = []
        for b in range(batch):
            Pt = torch.matrix_exp(Q[b] * t_obs[b])
            P_list.append(Pt)
        P = torch.stack(P_list)  # (batch, 4, 4)

        # Pick the observed transition probabilities P[i,j]
        probs = P[torch.arange(batch), i_obs, j_obs]
        probs = probs.clamp_min(self.eps)

        # Negative log-likelihood
        nll = -torch.log(probs)
        return nll.mean()

def ctmc_sequence_loss(q_pred, states, del_t, lengths, base_loss: CTMCMLELoss):
    '''
    シーケンス中の各遷移 (i->j, t) に対して CTMC NLL を計算し、平均を返す。
    q_pred : (B, 3)
    states : (B, 2, L)
    del_t  : (B, L)
    lengths: (B,)
    '''
    B, _, L = states.shape
    device = states.device

    mask = torch.arange(L, device=device).unsqueeze(0) < lengths.unsqueeze(1)
    i_obs = states[:, 0, :][mask]
    j_obs = states[:, 1, :][mask]
    t_obs = del_t[mask]

    # q_pred を各遷移に対応させる
    q_flat = q_pred.repeat_interleave(lengths, dim=0)
    return base_loss(q_flat, i_obs, j_obs, t_obs)


In [ ]:
def train_one_epoch(model, loader, optimizer, base_loss, device):
    model.train()
    total_loss = 0.0
    total_count = 0

    for states_batch, del_t_batch, targets_batch, lengths in loader:
        states_batch = states_batch.to(device)
        del_t_batch = del_t_batch.to(device)
        lengths = lengths.to(device)

        optimizer.zero_grad()
        q_pred = model(states_batch, del_t_batch, lengths)
        loss = ctmc_sequence_loss(q_pred, states_batch, del_t_batch, lengths, base_loss)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * states_batch.size(0)
        total_count += states_batch.size(0)

    return total_loss / max(total_count, 1)


def eval_one_epoch(model, loader, base_loss, device):
    model.eval()
    total_loss = 0.0
    total_count = 0
    with torch.no_grad():
        for states_batch, del_t_batch, targets_batch, lengths in loader:
            states_batch = states_batch.to(device)
            del_t_batch = del_t_batch.to(device)
            lengths = lengths.to(device)

            q_pred = model(states_batch, del_t_batch, lengths)
            loss = ctmc_sequence_loss(q_pred, states_batch, del_t_batch, lengths, base_loss)

            total_loss += loss.item() * states_batch.size(0)
            total_count += states_batch.size(0)
    return total_loss / max(total_count, 1)


In [ ]:
# ------------------------------------------------------------
# 1) CTMC MLE でゼロから学習
# ------------------------------------------------------------
base_loss = CTMCMLELoss()
model = DeepSets_varSets_forDiagnel(device=device).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)

best_val = float("inf")
patience_counter = 0
scratch_ckpt.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, base_loss, device)
    val_loss = eval_one_epoch(model, val_loader, base_loss, device)

    if val_loss < best_val:
        best_val = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), scratch_ckpt)
    else:
        patience_counter += 1

    print(f"[Scratch] Epoch {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f} | best {best_val:.4f} | patience {patience_counter}/{patience}")

    if patience_counter >= patience:
        print("Early stopping.")
        break

print(f"Best model saved to: {scratch_ckpt}")


In [ ]:
# ------------------------------------------------------------
# 2) 既存モデルを CTMC MLE でファインチューニング
# ------------------------------------------------------------
finetune_base_loss = CTMCMLELoss()
finetune_model = DeepSets_varSets_forDiagnel(device=device).to(device)

# 既存重みをロード
if pretrained_path.exists():
    state_dict = torch.load(pretrained_path, map_location=device)
    finetune_model.load_state_dict(state_dict)
    print(f"Loaded pretrained weights from {pretrained_path}")
else:
    print(f"Pretrained weights not found: {pretrained_path}. Training will start from scratch.")

finetune_optimizer = optim.Adam(finetune_model.parameters(), lr=lr * 0.1)  # 微調整は低めの LR

best_val = float("inf")
patience_counter = 0
finetune_ckpt.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(finetune_model, train_loader, finetune_optimizer, finetune_base_loss, device)
    val_loss = eval_one_epoch(finetune_model, val_loader, finetune_base_loss, device)

    if val_loss < best_val:
        best_val = val_loss
        patience_counter = 0
        torch.save(finetune_model.state_dict(), finetune_ckpt)
    else:
        patience_counter += 1

    print(f"[Finetune] Epoch {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f} | best {best_val:.4f} | patience {patience_counter}/{patience}")

    if patience_counter >= patience:
        print("Early stopping.")
        break

print(f"Finetuned model saved to: {finetune_ckpt}")
